In [ ]:
import sys

sys.path.append("workflow/scripts")

import _schemas
import _utils
import geopandas as gpd
import numpy as np
import pandas as pd
import prepare_glohydrores as prep


In [ ]:
raw_df = pd.read_csv("resources/automatic/glohydrores/data.csv")
raw_df.head()

# %%
hydro_df = _schemas.HydroSchema.validate(
    gpd.GeoDataFrame(
        {
            "powerplant_id": raw_df["ID"].apply(lambda x: "GloHydroRes_" + x),
            "name": raw_df["name"],
            "category": "hydropower",
            "technology": prep.get_technology(raw_df),
            "output_capacity_mw": raw_df["capacity_mw"],
            "start_year": raw_df["year"],
            "end_year": prep.get_end_year(raw_df),
            "status": prep.get_status(raw_df),
            "geometry": prep.get_geometry(raw_df, "plant_lon", "plant_lat"),
            "head_m": prep.get_head_m(raw_df),
            "reservoir_km3": raw_df["res_vol_km3"],
        }
    )
)
hydro_df[hydro_df["head_m"].isna()]

In [ ]:
HYDROLAKES_PATH = "resources/automatic/hydrolakes/HydroLAKES_polys_v10.gdb.zip"     # <-- adjust to your setup
hydrolakes = gpd.read_file(HYDROLAKES_PATH)[["geometry"]]
hydrolakes = hydrolakes.to_crs(hydro_df.crs)           # match CRS

# spatial join (vector-indexed)  → Boolean mask
hydro_df["in_lake"] = gpd.sjoin(
    hydro_df[["geometry"]],
    hydrolakes,
    how="left",
    predicate="within"
).index_right.notna()

print("Plants inside HydroLAKES polygons:",
      hydro_df["in_lake"].sum(), "/", len(hydro_df))

In [ ]:
plant = hydro_df.loc[200]
plant

In [ ]:
_utils.show_head_walk(plant)

In [ ]:
_utils.dem_src.name

In [ ]:
_utils.dem_src.close()